### Setup

In [3]:
# Library
import os
import torch
import json
from metric import * 
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore') 

# GPU
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device : ',device)   

# CONFIG
TEST_SIZE = 100

# PATH
CONFIG_PATH = '../config.json'
with open(CONFIG_PATH,'r') as f:
    config = json.load(f)
## DATA
SEED_IMAGE_FOLDER = config.get('SEED_IMAGE_FOLDER')
SEED_LABEL_FOLDER = config.get('SEED_LABEL_FOLDER')
SEED_LABEL_FILE = sorted(os.listdir(SEED_LABEL_FOLDER), key=lambda x: int(x.split('.')[0]))[:TEST_SIZE]
AUGEMNT_IMAGE_FOLDER = config.get('AUGEMNT_IMAGE_FOLDER')
AUGEMNT_LABEL_FOLDER =  config.get('AUGEMNT_LABEL_FOLDER')
AUGMENT_LABEL_FILE = sorted(os.listdir(AUGEMNT_LABEL_FOLDER), key=lambda x: int(x.split('.')[0]))[:TEST_SIZE]
PROMPT_FOLDER = config.get('PROMPT_FOLDER')
PROMPT_FILE = sorted(os.listdir(PROMPT_FOLDER), key=lambda x: int(x.split('.')[0]))[:TEST_SIZE]
ANSWER_IMAGE_FOLDER = config.get('ANSWER_IMAGE_FOLDER')
ANSWER_IMAGE_FILE = sorted(os.listdir(ANSWER_IMAGE_FOLDER), key=lambda x: int(x.split('.')[0]))[:TEST_SIZE]
SD_IMAGE_FOLDER = config.get('SD_IMAGE_FOLDER')
SD_IMAGE_FILE = sorted(os.listdir(SD_IMAGE_FOLDER), key=lambda x: int(x.split('.')[0]))[:TEST_SIZE]
FIGMA_IMAGE_FOLDER = config.get('FIGMA_IMAGE_FOLDER')
FIGMA_IMAGE_FILE = sorted(os.listdir(FIGMA_IMAGE_FOLDER), key=lambda x: int(x.split('.')[0]))[:TEST_SIZE]
## MODEL
PRE_TRAINED_MODEL_NAME="stablediffusionapi/deliberate-v2"
SAVE_WEIGHTS_PATH = '../Experiment/model_weights/FIGMA_weights_20250226_130326'

device :  cuda


### Augment Instruction Performace

- seed image - seed label 

In [2]:
image_folder = SEED_IMAGE_FOLDER
prompt_folder = SEED_LABEL_FOLDER
prompt_file = SEED_LABEL_FILE

clip_scores = []
for filename in tqdm(prompt_file):
    if filename.endswith('.json'):
        prompt_path = os.path.join(prompt_folder, filename)
        image_path = os.path.join(image_folder, filename.replace('.json', '.jpg'))

    with open(prompt_path, 'r') as f:
        prompt_data = json.load(f)
        persona = prompt_data.get("Prompt", {}).get("persona", "")
        task_description = prompt_data.get("Prompt", {}).get("task_description", "")
        constraint = prompt_data.get("Prompt", {}).get("constraint", "")
        caption = prompt_data.get("Input", {}).get("caption", "")
        add_info = prompt_data.get("Add_Info", "")
        
        final_prompt = " ".join([persona, task_description, constraint, caption, add_info])
        clip_score = calculate_clip_score(image_path, final_prompt)
        clip_scores.append(clip_score)
        
average_clip_score = sum(clip_scores) / len(clip_scores)
print(f"Average CLIP Score: {average_clip_score:.4f}")

100%|██████████| 100/100 [03:07<00:00,  1.87s/it]

Average CLIP Score: 0.2563


- augment image - augment label

In [3]:
image_folder = AUGEMNT_IMAGE_FOLDER
prompt_folder = AUGEMNT_LABEL_FOLDER
prompt_file = AUGMENT_LABEL_FILE

clip_scores = []
for filename in tqdm(prompt_file):
    if filename.endswith('.json'):
        prompt_path = os.path.join(prompt_folder, filename)
        image_path = os.path.join(image_folder, filename.replace('.json', '.jpg'))

    with open(prompt_path, 'r') as f:
        prompt_data = json.load(f)
        prompt = prompt_data.get("Prompt", "")
        caption = prompt_data.get("Input", {}).get("caption", "")
        add_info = prompt_data.get("Add_Info", "")
        
        final_prompt = " ".join([prompt, caption, add_info])
        clip_score = calculate_clip_score(image_path, final_prompt)
        clip_scores.append(clip_score)
        
average_clip_score = sum(clip_scores) / len(clip_scores)
print(f"Average CLIP Score: {average_clip_score:.4f}")

100%|██████████| 100/100 [03:34<00:00,  2.14s/it]

Average CLIP Score: 0.2944


### Generate Model Performance-Prompt Following Performance

- stablediffusion image - prompt

In [6]:
image_folder = SD_IMAGE_FOLDER
prompt_folder = PROMPT_FOLDER
prompt_file = PROMPT_FILE

clip_scores = []
for filename in tqdm(prompt_file):
    if filename.endswith('.json'):
        prompt_path = os.path.join(prompt_folder, filename)
        image_path = os.path.join(image_folder, filename.replace('.json', '.jpg'))

    with open(prompt_path, 'r') as f:
        prompt_data = json.load(f)
        final_prompt = prompt_data['summary']
        clip_score = calculate_clip_score(image_path, final_prompt)
        clip_scores.append(clip_score)
        
average_clip_score = sum(clip_scores) / len(clip_scores)
print(f"Average CLIP Score: {average_clip_score:.4f}")

 15%|█▌        | 15/100 [00:27<02:34,  1.82s/it]Error during conversion: ChunkedEncodingError(ProtocolError('Response ended prematurely'))
Error during conversion: ChunkedEncodingError(ProtocolError('Response ended prematurely'))
100%|██████████| 100/100 [03:01<00:00,  1.82s/it]

Average CLIP Score: 0.3386


- figma image - prompt

In [7]:
image_folder = FIGMA_IMAGE_FOLDER
prompt_folder = PROMPT_FOLDER
prompt_file = PROMPT_FILE

clip_scores = []
for filename in tqdm(prompt_file):
    if filename.endswith('.json'):
        prompt_path = os.path.join(prompt_folder, filename)
        image_path = os.path.join(image_folder, filename.replace('.json', '.jpg'))

    with open(prompt_path, 'r') as f:
        prompt_data = json.load(f)
        final_prompt = prompt_data['summary']
        clip_score = calculate_clip_score(image_path, final_prompt)
        clip_scores.append(clip_score)
        
average_clip_score = sum(clip_scores) / len(clip_scores)
print(f"Average CLIP Score: {average_clip_score:.4f}")

100%|██████████| 100/100 [03:03<00:00,  1.83s/it]

Average CLIP Score: 0.3184


### Generate Model Performance-Image Following Performance

- stablediffusion image - answer image

In [ ]:
fid_score_sd = calculate_fid(ANSWER_IMAGE_FOLDER,ANSWER_IMAGE_FILE,SD_IMAGE_FOLDER,SD_IMAGE_FILE)
lpips_sd = average_lpips(ANSWER_IMAGE_FOLDER, SD_IMAGE_FOLDER)
clip_similarity_sd = average_clip_similarity(ANSWER_IMAGE_FOLDER, SD_IMAGE_FOLDER)

In [7]:
print(f"SD's Average FID Score: {fid_score_sd:.4f}")
print(f"SD's Average LPIPS Score: {lpips_sd:.4f}")
print(f"SD's Average CLIP Similarity: {clip_similarity_sd:.4f}")

SD's Average FID Score: 5.2137
SD's Average LPIPS Score: 0.5562
SD's Average CLIP Similarity: 0.7378


- figma image - answer image

In [ ]:
fid_score_figma = calculate_fid(ANSWER_IMAGE_FOLDER,ANSWER_IMAGE_FILE,FIGMA_IMAGE_FOLDER,FIGMA_IMAGE_FILE)
lpips_figma = average_lpips(ANSWER_IMAGE_FOLDER, FIGMA_IMAGE_FOLDER)
clip_similarity_figma = average_clip_similarity(ANSWER_IMAGE_FOLDER, SD_IMAGE_FOLDER)

In [5]:
print(f"FIGMA's Average FID Score: {fid_score_figma:.4f}")
print(f"FIGMA's Average LPIPS Score: {lpips_figma:.4f}")
print(f"FIGMA's Average CLIP Similarity: {clip_similarity_figma:.4f}")

FIGMA's Average FID Score: 5.8132
FIGMA's Average LPIPS Score: 0.5397
FIGMA's Average CLIP Similarity: 0.7378
